In [1]:
%cd /mnt/data/symbolic-regression-dst-public-repo

/mnt/data/symbolic-regression-dst-public-repo


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from tqdm import tqdm
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr
from sympy.printing import latex
import os

# Internal module imports
import utils
import storm_dates
import metrics
import baseline_models

# from evaluation_engine import UnifiedModel, simulate_storm, compute_features
from evaluation_engine import EquationModel, simulate_storm
from train_script import load_and_preprocess, compute_features

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [3]:
FEATURES = ["Vp", "Np", "Bzsouth", "Bmag", "By", "DST"]
MODE = 'template'  # 'template' or 'default'
OUTPUT_DIR = "test_plots_notebook_asdf"
RAW_EQ = 'g = ((#3 - -1.6177002) * ((#3 * #2) + #1)) * -0.0018822174; d = square((#1 * 0.01634224) + -1.1816965)'

output_folder = OUTPUT_DIR
# Count number of existing subfolders
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [4]:
raw_data = load_and_preprocess()
data = compute_features(raw_data)

Reading from file ./data/all_timeline/ace_imf_1h_1998.csv
Reading from file ./data/all_timeline/ace_imf_1h_1999.csv
Reading from file ./data/all_timeline/ace_imf_1h_2000.csv
Reading from file ./data/all_timeline/ace_imf_1h_2001.csv
Reading from file ./data/all_timeline/ace_imf_1h_2002.csv
Reading from file ./data/all_timeline/ace_imf_1h_2003.csv
Reading from file ./data/all_timeline/ace_imf_1h_2004.csv
Reading from file ./data/all_timeline/ace_imf_1h_2005.csv
Reading from file ./data/all_timeline/ace_imf_1h_2006.csv
Reading from file ./data/all_timeline/ace_imf_1h_2007.csv
Reading from file ./data/all_timeline/ace_imf_1h_2008.csv
Reading from file ./data/all_timeline/ace_imf_1h_2009.csv
Reading from file ./data/all_timeline/ace_imf_1h_2010.csv
Reading from file ./data/all_timeline/ace_imf_1h_2011.csv
Reading from file ./data/all_timeline/ace_imf_1h_2012.csv
Reading from file ./data/all_timeline/ace_imf_1h_2013.csv
Reading from file ./data/all_timeline/ace_imf_1h_2014.csv
Reading from f

In [5]:
def predict_and_plot_storm(model, start, end, storm_df, storm_id, save_path):
    # 1. Generate Predictions
    y_true = storm_df[start:end]["DST"].values

    res_eq = simulate_storm(model, storm_df)
    res_burton = baseline_models.burton_prediction(storm_df)
    res_obm = baseline_models.obm_prediction(storm_df)
    ddm1 = baseline_models.ddm1_prediction(storm_df)
    ddm2 = baseline_models.ddm2_prediction(storm_df)
    ddm3 = baseline_models.ddm3_prediction(storm_df)

    res_eq = res_eq[start:end]["DST_pred"].values
    res_burton = res_burton[start:end]["DST_pred"].values
    res_obm = res_obm[start:end]["DST_pred"].values
    res_ddm1 = ddm1[start:end]["DST_pred"].values
    res_ddm2 = ddm2[start:end]["DST_pred"].values
    res_ddm3 = ddm3[start:end]["DST_pred"].values

    # 2. Calculate Metrics
    m_eq = baseline_models.get_all_metrics(y_true, res_eq)
    m_burton = baseline_models.get_all_metrics(y_true, res_burton)
    m_obm = baseline_models.get_all_metrics(y_true, res_obm)
    m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
    m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
    m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

    # 3. Setup Figure (3 Columns)
    fig, axs = plt.subplots(1, 3, figsize=(24, 7), constrained_layout=True)
    fig.suptitle(
        rf"Evaluation for Equation: ${model.latex_str()}$", fontsize=16, wrap=True
    )
    # Column 1: Time Series
    axs[0].plot(
        storm_df[start:end].index,
        y_true,
        color="black",
        label="Observed",
        alpha=0.6,
        linewidth=2,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_eq,
        color="blue",
        linestyle="--",
        label="Equation",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_burton,
        color="yellow",
        linestyle="--",
        label="Burton",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_obm,
        color="green",
        linestyle="--",
        label="OBM",
        linewidth=1.5,
    )
    
    axs[0].plot(
        storm_df[start:end].index,
        res_ddm1,
        color="orange",
        linestyle="--",
        label="DDM1",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_ddm2,
        color="purple",
        linestyle="--",
        label="DDM2",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_ddm3,
        color="cyan",
        linestyle="--",
        label="DDM3",
        linewidth=1.5,
    )
    
    axs[0].set_title(f"Storm {storm_id} Reconstruction")
    axs[0].legend()
    axs[0].grid(True)
    axs[0].set_xlim(start, end)

    # Column 2: Prediction Error
    diff_eq = res_eq - y_true
    diff_burton = res_burton - y_true
    diff_obm = res_obm - y_true
    diff_ddm1 = res_ddm1 - y_true
    diff_ddm2 = res_ddm2 - y_true
    diff_ddm3 = res_ddm3 - y_true


    axs[1].plot(storm_df[start:end].index, diff_eq, color="blue", label="Eq Error")
    axs[1].plot(
        storm_df[start:end].index, diff_burton, color="yellow", label="Burton Error"
    )
    axs[1].plot(storm_df[start:end].index, diff_obm, color="green", label="OBM Error")
    axs[1].plot(storm_df[start:end].index, diff_ddm1, color="orange", label="DDM1 Error")
    axs[1].plot(storm_df[start:end].index, diff_ddm2, color="purple", label="DDM2 Error")
    axs[1].plot(storm_df[start:end].index, diff_ddm3, color="cyan", label="DDM3 Error")
    axs[1].axhline(0, color="black", linestyle="--")

    title_metrics = (
        f"Error Comparison\n"
        f"Eq: RMSE {m_eq[0]:.2f} | MAE {m_eq[1]:.2f} | R2 {m_eq[2]:.2f} | CC {m_eq[3]:.2f} | BFE {m_eq[4]:.2f}\n"
        f"Burton: RMSE {m_burton[0]:.2f} | MAE {m_burton[1]:.2f} | R2 {m_burton[2]:.2f} | CC {m_burton[3]:.2f} | BFE {m_burton[4]:.2f}\n"
        f"OBM: RMSE {m_obm[0]:.2f} | MAE {m_obm[1]:.2f} | R2 {m_obm[2]:.2f} | CC {m_obm[3]:.2f} | BFE {m_obm[4]:.2f}\n"
        f"DDM1: RMSE {m_ddm1[0]:.2f} | MAE {m_ddm1[1]:.2f} | R2 {m_ddm1[2]:.2f} | CC {m_ddm1[3]:.2f} | BFE {m_ddm1[4]:.2f}\n"
        f"DDM2: RMSE {m_ddm2[0]:.2f} | MAE {m_ddm2[1]:.2f} | R2 {m_ddm2[2]:.2f} | CC {m_ddm2[3]:.2f} | BFE {m_ddm2[4]:.2f}\n"
        f"DDM3: RMSE {m_ddm3[0]:.2f} | MAE {m_ddm3[1]:.2f} | R2 {m_ddm3[2]:.2f} | CC {m_ddm3[3]:.2f} | BFE {m_ddm3[4]:.2f}"
    )
    axs[1].set_title(title_metrics, fontsize=9)
    axs[1].set_ylabel("Error (nT)")
    axs[1].legend()
    axs[1].grid(True)
    axs[1].set_xlim(start, end)

    # Column 3: BFE
    baseline_models.plot_evaluation_bfe_multi(
        axs[2],
        y_true,
        [res_eq, res_burton, res_obm, res_ddm1, res_ddm2, res_ddm3],
        ["Equation", "Burton", "OBM", "DDM1", "DDM2", "DDM3"],
        ["blue", "yellow", "green", "orange", "purple", "cyan"],
    )

    plt.savefig(save_path)
    plt.close()


In [6]:
def save_prediction_data(model, start, end, storm_df, output_path):
    """
    Generates and saves a CSV with observed and predicted DST and dDST/dt.
    """
    # 1. Observed Data
    # Real dDST is calculated as the difference to the next hour
    real_dst = storm_df[start:end]["DST"].values
    real_ddst = storm_df[start:end]["DST"].diff().shift(-1).values

    # 2. Equation Predictions
    # We need the iterative predictions for DST
    pred_dst_eq = simulate_storm(model, storm_df)
    if model.is_template:
        pred_dst_eq = pred_dst_eq[start:end][
            ["DST_pred", "dDST", "injection_component", "decay_component"]
        ]
    else:
        pred_dst_eq = pred_dst_eq[start:end][["DST_pred", "dDST"]]
    # 3. Baseline Predictions (Burton & OBM)
    pred_dst_burton = baseline_models.burton_prediction(storm_df)
    pred_dst_burton = pred_dst_burton[start:end][["DST_pred", "dDST"]]
    pred_dst_burton.columns = ["DST_pred_burton", "dDST_burton"]
    pred_dst_burton = pred_dst_burton[start:end][["DST_pred_burton", "dDST_burton"]]
    pred_dst_obm = baseline_models.obm_prediction(storm_df)
    pred_dst_obm = pred_dst_obm[start:end][["DST_pred", "dDST"]]
    pred_dst_obm.columns = ["DST_pred_obm", "dDST_obm"]
    pred_dst_obm = pred_dst_obm[start:end][["DST_pred_obm", "dDST_obm"]]
    pred_dst_ddm1 = baseline_models.ddm1_prediction(storm_df)    
    pred_dst_ddm1.columns = ["DST_pred_ddm1", "dDST_ddm1"]
    pred_dst_ddm1 = pred_dst_ddm1[start:end][["DST_pred_ddm1", "dDST_ddm1"]]
    pred_dst_ddm2 = baseline_models.ddm2_prediction(storm_df)
    pred_dst_ddm2.columns = ["DST_pred_ddm2", "dDST_ddm2"]
    pred_dst_ddm2 = pred_dst_ddm2[start:end][["DST_pred_ddm2", "dDST_ddm2"]]
    pred_dst_ddm3 = baseline_models.ddm3_prediction(storm_df)
    pred_dst_ddm3.columns = ["DST_pred_ddm3", "dDST_ddm3"]
    pred_dst_ddm3 = pred_dst_ddm3[start:end][["DST_pred_ddm3", "dDST_ddm3"]]


    # 4. Construct Comprehensive DataFrame

    if model.is_template:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,
                "Pred_DST_Equation": pred_dst_eq["DST_pred"].values,
                "Pred_dDST_dt_Equation": pred_dst_eq["dDST"].values,
                "Injection_Component": pred_dst_eq["injection_component"].values,
                "Decay_Component": pred_dst_eq["decay_component"].values,
                "Pred_DST_Burton": pred_dst_burton["DST_pred_burton"].values,
                "Pred_dDST_dt_Burton": pred_dst_burton["dDST_burton"].values,
                "Pred_DST_OBM": pred_dst_obm["DST_pred_obm"].values,
                "Pred_dDST_dt_OBM": pred_dst_obm["dDST_obm"].values,
                "Pred_DST_DDM1": pred_dst_ddm1["DST_pred_ddm1"].values,
                "Pred_dDST_dt_DDM1": pred_dst_ddm1["dDST_ddm1"].values,
                "Pred_DST_DDM2": pred_dst_ddm2["DST_pred_ddm2"].values,
                "Pred_dDST_dt_DDM2": pred_dst_ddm2["dDST_ddm2"].values,
                "Pred_DST_DDM3": pred_dst_ddm3["DST_pred_ddm3"].values,
                "Pred_dDST_dt_DDM3": pred_dst_ddm3["dDST_ddm3"].values,
            }
        ).set_index("Timestamp")
    else:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,
                "Pred_DST_Equation": pred_dst_eq["DST_pred"].values,
                "Pred_dDST_dt_Equation": pred_dst_eq["dDST"].values,
                "Pred_DST_Burton": pred_dst_burton["DST_pred_burton"].values,
                "Pred_dDST_dt_Burton": pred_dst_burton["dDST_burton"].values,
                "Pred_DST_OBM": pred_dst_obm["DST_pred_obm"].values,
                "Pred_dDST_dt_OBM": pred_dst_obm["dDST_obm"].values,
                "Pred_DST_DDM1": pred_dst_ddm1["DST_pred_ddm1"].values,
                "Pred_dDST_dt_DDM1": pred_dst_ddm1["dDST_ddm1"].values,
                "Pred_DST_DDM2": pred_dst_ddm2["DST_pred_ddm2"].values,
                "Pred_dDST_dt_DDM2": pred_dst_ddm2["dDST_ddm2"].values,
                "Pred_DST_DDM3": pred_dst_ddm3["DST_pred_ddm3"].values,
                "Pred_dDST_dt_DDM3": pred_dst_ddm3["dDST_ddm3"].values,

            }
        ).set_index("Timestamp")

    results_df.to_csv(output_path)
    return results_df

In [7]:
storms = []

model = EquationModel(RAW_EQ, FEATURES, is_template=MODE == "template")
test_storms = storm_dates.TEST_STORMS_SYMBOLIC_REGRESSION
for sd, ed, storm_id in tqdm(test_storms):
    start = pd.to_datetime(sd)
    end = pd.to_datetime(ed)
    storm_df = data[
        start - pd.DateOffset(hours=1) : end + pd.DateOffset(hours=1)
    ].copy()
    if storm_df.empty:
        continue

    file_name = f"storm_{storm_id}.png"
    predict_and_plot_storm(
        model,
        start,
        end,
        storm_df,
        storm_id,
        os.path.join(OUTPUT_DIR, file_name),
    )

    csv_name = f"data_storm_{storm_id}.csv"
    storms.append(
        save_prediction_data(
            model, start, end, storm_df, os.path.join(OUTPUT_DIR, csv_name)
        )
    )
    
with open(os.path.join(OUTPUT_DIR, 'equation.txt'), 'w') as f:
    f.write(f'Equation: {RAW_EQ}\n')            
    f.write(f'LaTeX: {latex(model.latex_str())}\n')

  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 20/20 [00:12<00:00,  1.59it/s]


In [8]:
metrics = ["RMSE", "MAE", "R2", "CC", "BFE"]
equations = ["Equation", "Burton", "OBM", "DDM1", "DDM2", "DDM3"]

columns = [f"{eq}_{metric}" for eq in equations for metric in metrics]

summary_df = pd.DataFrame(
    columns=["Storm Index"] + columns,
)

for storm_index, storm in enumerate(storms):
    y_true = storm["Observed_DST"].values
    res_eq = storm["Pred_DST_Equation"].values
    res_burton = storm["Pred_DST_Burton"].values
    res_obm = storm["Pred_DST_OBM"].values
    res_ddm1 = storm["Pred_DST_DDM1"].values
    res_ddm2 = storm["Pred_DST_DDM2"].values
    res_ddm3 = storm["Pred_DST_DDM3"].values

    m_eq = baseline_models.get_all_metrics(y_true, res_eq)
    m_burton = baseline_models.get_all_metrics(y_true, res_burton)
    m_obm = baseline_models.get_all_metrics(y_true, res_obm)
    m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
    m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
    m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

    summary_df.loc[len(summary_df)] = [
        storm_index,
        *m_eq,
        *m_burton,
        *m_obm,
        *m_ddm1,
        *m_ddm2,
        *m_ddm3,
    ]

summary_df.loc[len(summary_df)] = ["Mean", *summary_df[columns].mean().values]

global_data = pd.concat(storms, ignore_index=True)
y_true = global_data["Observed_DST"].values
res_eq = global_data["Pred_DST_Equation"].values
res_burton = global_data["Pred_DST_Burton"].values
res_obm = global_data["Pred_DST_OBM"].values
res_ddm1 = global_data["Pred_DST_DDM1"].values
res_ddm2 = global_data["Pred_DST_DDM2"].values
res_ddm3 = global_data["Pred_DST_DDM3"].values

m_eq = baseline_models.get_all_metrics(y_true, res_eq)
m_burton = baseline_models.get_all_metrics(y_true, res_burton)
m_obm = baseline_models.get_all_metrics(y_true, res_obm)
m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

summary_df.loc[len(summary_df)] = [
    "Global",
    *m_eq,
    *m_burton,
    *m_obm,
    *m_ddm1,
    *m_ddm2,
    *m_ddm3,
]

display(summary_df)

,Storm Index,Equation_RMSE,Equation_MAE,Equation_R2,Equation_CC,Equation_BFE,Burton_RMSE,Burton_MAE,Burton_R2,Burton_CC,...,DDM2_RMSE,DDM2_MAE,DDM2_R2,DDM2_CC,DDM2_BFE,DDM3_RMSE,DDM3_MAE,DDM3_R2,DDM3_CC,DDM3_BFE
0,0.0,9.588300,6.500370,0.773095,0.930841,13.207819,12.546739,9.426567,0.611471,0.804031,...,17.034353,14.718489,0.283836,0.819670,19.285046,17.429228,14.902320,0.250248,0.849179,20.944809
1,1.0,15.192461,11.851948,0.809246,0.915864,16.569627,23.402983,17.430162,0.547353,0.869635,...,20.994874,17.672795,0.635713,0.870809,22.088199,22.442717,18.173790,0.583737,0.864456,25.386413
2,2.0,11.237450,8.654356,0.722794,0.875983,13.280423,22.444927,16.827487,-0.105866,0.856593,...,11.710892,9.329760,0.698945,0.874185,13.104050,11.378825,9.160091,0.715776,0.872556,12.828221
3,3.0,10.044948,7.555612,0.743478,0.902024,8.783040,19.062180,15.620942,0.076206,0.792714,...,8.303250,6.470139,0.824723,0.917542,11.441876,8.670891,6.332414,0.808857,0.904447,13.883947
4,4.0,8.511096,6.053109,0.824969,0.922277,14.349803,17.059837,14.234517,0.296775,0.867234,...,11.990044,9.586551,0.652635,0.926882,19.031650,11.862568,8.934847,0.659982,0.920503,19.970899
5,5.0,18.298826,15.643581,0.662809,0.925201,12.112950,27.156568,23.604603,0.257358,0.894657,...,10.840170,8.734074,0.881668,0.955437,13.497036,12.354960,9.578556,0.846286,0.954712,16.759531
6,6.0,16.281749,13.651869,0.810158,0.962543,16.951827,31.161847,25.739701,0.304599,0.930523,...,13.320289,10.330457,0.872938,0.963738,17.936865,14.717364,11.742180,0.844887,0.954268,19.772733
7,7.0,12.295071,8.911478,0.909878,0.961130,17.806137,43.516911,23.133641,-0.128974,0.898528,...,22.490189,16.826801,0.698454,0.883757,30.616512,20.468681,15.214925,0.750226,0.886201,30.187252
8,8.0,12.746520,10.872018,0.622271,0.829288,12.647117,24.764109,20.315994,-0.425746,0.715543,...,10.405858,7.936314,0.748260,0.888742,14.058279,11.805307,9.352887,0.675995,0.859124,16.881751
9,9.0,12.545221,8.403906,0.885656,0.961494,18.279292,16.754713,13.386004,0.796046,0.900956,...,20.014049,16.071049,0.708977,0.932586,28.117249,22.085126,17.377477,0.645630,0.929705,32.083276
